## Building a Neural Network with 1 Hidden Layer from scratch using Numpy and scipy

This neural network will have:
1. Input Layer with 3 neurons
2. Hidden Layer with 2 neurons
3. Output layer with 1 neuron

Properties of the neural network: 
- Hidden layers will have sigmoid and relu functions as the activations with backpropagation. 
- Loss function cross entropy loss for classification and MSELoss for regression will also be coded from scratch.
- Optimizers optim.SGD or the Stochastic Gradient Descent and optim.Adam or the Adam will be coded from scratch as well. 
- Finally lets compare this with torch for time taken
- Lastly, converting them into ONNX by hand using protobuf will also be properly done so that they can be displayed in netron.app

### We will follow an OOP structure with maintaining them as different submodules for different classes. First, lets start with working on a Neuron and training it

In [1]:
from typing import Optional, List, Any, Tuple, Union
import torch
import pandas as pd
import numpy as np
import scipy.linalg as linal

In [2]:
class MatMulError(Exception):
    def __init__(self,message):
        self.message = message
        super().__init__(message)
    def __str__(self):
        return f"{self.message} has been raised when performing Matrix Multiplication"
    
class DimensionMisMatchError(Exception):
    def __init__(self,message):
        self.message = message 
        super().__init__(message)
    def __str__(self):
        return f"{self.message} has been raised when checking for dimension match"

In [3]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_dash(x):
    return sigmoid(x)*(1-sigmoid(x))

def cross_entropy_loss(y_true, y_pred, reduction: str = 'mean'):
    """ 
    Produces the mean or sum of the Cross Entropy Loss over the batch
    """
    # naive logic
    # true_idx_1 = np.where(y_true==0)[0]
    # true_idx_2 = np.where(y_true==1)[0]
    # loss = -(y_true[true_idx_2]*np.log(y_pred[true_idx_2])).sum()-((1-y_true[true_idx_1])*np.log(1 - y_pred[true_idx_1])).sum() 

    y_pred = np.clip(y_pred, 1e-15, 1-1e-15) # to keep the array from having 0 or 1 values so that the log might explode or cause NaN values
    loss = -np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    if reduction == 'mean':
        return loss.sum()/y_true.shape[0]
    else:
        return loss.sum()

def cross_entropy_loss_derivative(y_true, y_pred):
    return (y_pred-y_true)

def accuracy(y_true, y_pred):
    return float((y_true.reshape(-1,1)==y_pred.reshape(-1,1)).sum()/y_pred.shape[0])*100

In [4]:
class Neuron:
    def __init__(self, in_features:int):
        self.in_f = in_features 
        self.w = np.random.randn(self.in_f,1) # m x 1 weight matrix for m features and 1 output value
        self.b = np.random.random() # 1 x 1 bias matrix for the neuron

    def fit(self,X: Union[np.array, torch.tensor, pd.DataFrame], y: Union[np.array, torch.tensor, pd.DataFrame], 
            n_epochs:int = 100, lr:float = 1e-4, verbose: bool = False, 
            frequency: int = 5):
        """ 
        Function to fit the given dataset with the Neuron's Sigmoid function and train according to the given hyperparameters.
        Uses Standard Gradient Descent.
        """
        Y = y.reshape(-1,1) # to make the dimensions n x 1 as a 2D matrix instead of a 1D array
        for epoch in range(n_epochs):
            y_pred = self.predict_proba(X)
            loss = cross_entropy_loss(Y,y_pred)
            loss_der = cross_entropy_loss_derivative(Y,y_pred)
            w = self.w - lr*(X.T)@(loss_der)
            b = self.b - lr*np.sum(loss_der)/X.shape[0]
            self.w = w
            self.b = b
            if epoch%frequency == 0 or verbose:
                print(f"Epoch: {epoch}, loss: {loss}")

    def predict(self, X: Union[np.array, torch.tensor, pd.DataFrame]):
        if len(X.shape) > 2:
            raise DimensionMisMatchError(f"X is 3 dimensional with {X.shape}, expected 2 dimensional.")
        if not self.w.shape[0] in X.shape:
            raise MatMulError(f"Shape mismatch: Matrices with sizes {self.w.shape} cannot be multiplied with {X.shape}") 
        try:
            if self.w.shape[0] == X.shape[1]:
                return np.array(sigmoid(X@self.w+self.b) >=0.5, dtype=float) # returns n x 1 array
            elif self.w.shape[0] == X.shape[0]:
                return np.array(sigmoid(X.T@self.w+self.b) >=0.5, dtype=float) # return n x 1 array
        except Exception as e:
            print(f"Receiving error : {e}")
            return None
    
    def predict_proba(self, X: Union[np.array, torch.tensor, pd.DataFrame]):
        if len(X.shape) > 2:
            raise DimensionMisMatchError(f"X is 3 dimensional with {X.shape}, expected 2 dimensional.")
        if not self.w.shape[0] in X.shape:
            raise MatMulError(f"Shape mismatch: Matrices with sizes {self.w.shape} cannot be multiplied with {X.shape}") 
        try:
            if self.w.shape[0] == X.shape[1]:
                return sigmoid(X@self.w+self.b) # returns n x 1 array
            elif self.w.shape[0] == X.shape[0]:
                return sigmoid(X.T@self.w+self.b) # return n x 1 array
        except Exception as e:
            print(f"Receiving error : {e}")
            return None

#### Lets generate a sample dataset with gaussian noise

In [5]:
X = np.random.randn(1000,6)
y = np.array([0,1]*500)
np.random.shuffle(y)
neuron = Neuron(6)

In [6]:
neuron.fit(X,y,100,0.001,False)

Epoch: 0, loss: 1.080401446258235
Epoch: 5, loss: 0.8382047823125
Epoch: 10, loss: 0.7887549435706037
Epoch: 15, loss: 0.7822426764168877
Epoch: 20, loss: 0.781283171878581
Epoch: 25, loss: 0.7809857973752983
Epoch: 30, loss: 0.7807675999891314
Epoch: 35, loss: 0.7805592252268183
Epoch: 40, loss: 0.7803523937792909
Epoch: 45, loss: 0.7801461150318794
Epoch: 50, loss: 0.7799402698727503
Epoch: 55, loss: 0.7797348434678629
Epoch: 60, loss: 0.7795298334608872
Epoch: 65, loss: 0.7793252389882067
Epoch: 70, loss: 0.7791210593648757
Epoch: 75, loss: 0.7789172939275397
Epoch: 80, loss: 0.7787139420156862
Epoch: 85, loss: 0.7785110029694136
Epoch: 90, loss: 0.7783084761291684
Epoch: 95, loss: 0.77810636083572


In [7]:
accuracy(neuron.predict(X),y)

50.0

In [8]:
X = np.random.randn(10000,8)
y = np.array((5*X**2+23*X+np.random.sample((X.shape))).sum(axis=1) > 38,dtype=float) 

In [9]:
neuron = Neuron(8)
neuron.fit(X,y,1000,1e-5,False,frequency=50)

Epoch: 0, loss: 1.247322351491632
Epoch: 50, loss: 0.5481714269663633
Epoch: 100, loss: 0.3448428527089409
Epoch: 150, loss: 0.28754147400965846
Epoch: 200, loss: 0.2660560772933944
Epoch: 250, loss: 0.25491875505720074
Epoch: 300, loss: 0.24768367444782674
Epoch: 350, loss: 0.24236634705693072
Epoch: 400, loss: 0.23820165128325863
Epoch: 450, loss: 0.2348226065301374
Epoch: 500, loss: 0.2320195989804385
Epoch: 550, loss: 0.22965776433791088
Epoch: 600, loss: 0.22764355380639467
Epoch: 650, loss: 0.22590889352343538
Epoch: 700, loss: 0.2244026167410077
Epoch: 750, loss: 0.22308534493372909
Epoch: 800, loss: 0.22192620714061012
Epoch: 850, loss: 0.22090063171283164
Epoch: 900, loss: 0.21998880645900007
Epoch: 950, loss: 0.21917457578980276


In [10]:
y_pred = neuron.predict(X)
accuracy(y, y_pred)

90.02

In [11]:
(y.reshape(-1,1) != y_pred).sum()

np.int64(998)

### Coding the Neural Network
- Handwritten gradients
- Next part: autograd from scratch

In [2]:
import numpy as np
np.random.random(4)

array([0.29879414, 0.88425199, 0.09631557, 0.31906443])

In [ ]:
class Unit(Neuron):
    def __init__(self,in_features: int, out_features:int):
        super().__init__(in_features, out_features)

class NeuralNetwork:
    def __init__(self,in_features,hidden_units,out_features,)